# 세그먼트 트리 (Segment Tree)

`-` 배열의 원소 갱신과 구간 쿼리를 효율적으로 처리하는 자료 구조

`-` 연산이 [모노이드](https://en.wikipedia.org/wiki/Monoid)여야 활용할 수 있다 (결합법칙을 만족하고 항등원이 존재하는 이항 연산)

`-` 구현체: https://github.com/Jaesu26/python-cp-templates/blob/main/src/segment-tree.py

## 타이어 팔기

- 문제 출처: [정올 2790번](https://jungol.co.kr/problem/2790)

`-` 바깥지름이 큰 타이어순으로 고려하자. 자신의 바깥지름보다 안지름이 큰 중첩된 타이어 중 가격 합이 최대인 걸 골라 중첩시키면 된다

`-` 자신의 바깥지름보다 안지름이 큰 타이어는 당연히 바깥지름도 더 크므로 전이 상태가 꼬이거나 하지는 않다

`-` 좌표 압축 후 세그먼트 트리를 사용하면 $O(N \log N)$에 문제를 해결할 수 있다

In [3]:
def compress(array):
    sorted_unique = sorted(set(array))
    ranks = {a: i for i, a in enumerate(sorted_unique)}
    return ranks


def solution():
    N = int(input())
    array = [list(map(int, input().split())) for _ in range(N)]
    array.sort(key=lambda x: (-x[1], -x[0]))
    lengths = [I for I, _, _ in array] + [O for _, O, _ in array]
    ranks = compress(lengths)
    m = len(ranks)
    tree = SegmentTree(m, max, 0)
    query = tree.query
    update = tree.update
    get = tree.get
    for I, O, P in array:
        i = ranks[I]
        o = ranks[O]
        mx = query(o, m - 1)
        p = get(i)
        if p < mx + P:
            update(i, mx + P)
    answer = query(0, m - 1)
    print(answer)


solution()

# input
# 1
# 1 2 3

 1
 1 2 3


3


`-` 세그먼트 트리 구현 코드는 앞으로 생략한다 (계속 쓸거라 코드 길이만 길어짐) 

## 컨베이어 벨트

- 문제 출처: [정올 6289번](https://jungol.co.kr/problem/6289)

`-` 구간질의 단계별 문제이다

`-` 나이브하게 처리하면 $O(NM)$이다. 대신, 박스의 여유 무게별 인덱스들을 배열로 관리하자. 그리고 박스의 여유 무게별로 가장 작은 인덱스를 세그먼트 트리로 관리하자. 그럼 구간 쿼리로 물건을 담을 수 있는 가장 가까운 박스를 알 수 있다

`-` 그런데 갱신을 위해선 배열의 적절한 위치에 원소를 빠르게 삽입할 수 있어야 하는데 이를 위해선 레드 블랙 트리 같은 게 필요하다. 하지만 파이썬에는 이를 수행하는 내장 모듈이 없다...

`-` 여기서 진전이 안 되서 태그를 봤는데 우선순위 큐가 있었다. 위에서 적절한 위치에 원소를 삽입해야 한다고 했다. 근데 사실 필요한 건 최솟값뿐이며 나머지는 중요하지 않다. 항상 올바른 최솟값을 얻을 수 있기만 하면 된다

`-` 따라서 무게별 인덱스들을 우선순위 큐로 관리할 수 있으니 레드 블랙 트리 같은 건 필요 없다

`-` 추가로 좌표 압축이 필요한데 오프라인으로 처리를 못해서 대신 다이나믹 세그먼트 트리를 사용하자

`-` 하지만 아무리 최적화를 해도 TLE가 발생했고 다른 풀이는 생각이 안 나서 제미나이한테 힌트를 받았다

`-` 박스의 여유 무게의 최댓값을 기록하는 세그먼트 트리를 만든 뒤 쿼리가 들어오면 루트 노드부터 시작해 왼쪽 자식과 오른쪽 자식을 비교하며 내려가면 $O(\log N)$에 해결할 수 있다

`-` 좌표 압축을 할 필요도 없고 다이나믹 세그먼트 트리를 쓸 필요도 없다. 전체 알고리즘의 시간 복잡도는 $O(N + M\log N)$이다

In [7]:
def query(tree, w):
    node = 1
    if tree._tree[node] < w:
        return -1
    while node < tree._size:
        l = node << 1
        r = l | 1
        if tree._tree[l] >= w:
            node = l
        else:
            node = r
    return node - tree._size


def solution():
    N, M = map(int, input().split())
    limits = list(map(int, input().split()))
    weights = list(map(int, input().split()))
    tree = SegmentTree(limits, max, 0)
    answers = [0] * M
    for i, w in enumerate(weights):
        near = query(tree, w)
        if near == -1:
            answers[i] = 0
            continue
        answers[i] = near + 1
        r = tree.get(near)
        tree.update(near, r - w)
    print(*answers)


solution()

# input
# 2 2
# 5 10
# 7 3

 2 2
 5 10
 7 3


2 1


`-` 보통은 세그먼트 트리에서 노드(배열 인덱스도 포함(리프 노드))를 이용해 특정 값을 효율적으로 계산하나 반대로 값을 이용해 조건을 만족하는 노드(리프 노드를 가리키는 배열의 인덱스)를 효율적으로 찾을 수도 있다. 이 문제는 후자의 경우이다

## 탭댄스(tap)

- 문제 출처: [정올 2438번](https://jungol.co.kr/problem/2438)

`-` 세그먼트 트리 응용 문제이다. 백준에서 풀어봤다. 연산이 특이할 뿐 모노이드이므로 세그먼트 트리 템플릿을 그대로 사용할 수 있다 (이를 보여주고자 풀이를 기록)

`-` 노드마다 (점수, 구간의 첫 문자를 포함하는 최대 점수, 구간의 끝 문자를 포함하는 최대 점수, 구간의 첫 문자, 구간의 끝 문자, 구간의 길이)를 저장하자

`-` 두 노드를 병합하면 기본적으로 $m = (\max(l[0], r[0]),\, l[1], \, r[2],\, l[3],\,  r[4], \, l[5] + r[5])$이다

`-` 만약 $l[4] \ne r[3]$라면 $m[0] = \max(l[2] + r[1], m[0])$이다. 여기에 추가로 $l[1] = l[5]$라면 $m[1] = l[5] + r[1]$이고 $r[2] = r[5]$라면 $m[2] = l[2] + r[5]$이다

`-` 항등원은 None으로 설정하여 연산에서 두 노드 중 하나가 항등원일시 나머지 하나를 반환하도록 구현하면 된다

In [30]:
def merge(l, r):
    if l is None:
        return r
    if r is None:
        return l
    m = [max(l[0], r[0]), l[1], r[2], l[3], r[4], l[5] + r[5]]
    if l[4] == r[3]:
        return m
    m[0] = max(l[2] + r[1], m[0])
    if l[1] == l[5]:
        m[1] = l[5] + r[1]
    if r[2] == r[5]:
        m[2] = l[2] + r[5]
    return m          


def solution():
    N, Q = map(int, input().split())
    queries = [int(input()) - 1 for _ in range(Q)]
    array = [[1, 1, 1, 0, 0, 1] for _ in range(N)]
    tree = SegmentTree(array, merge, None)
    answers = [0] * Q
    for i, q in enumerate(queries):
        leaf = tree.get(q)
        leaf[3] = leaf[4] = leaf[3] ^ 1
        tree.update(q, leaf)
        answers[i] = tree._tree[1][0]
    print("\n".join(map(str, answers)))


solution()

# input
# 5 2
# 2
# 4

 5 2
 2
 4


3
5
